In [ ]:
import pandas as pd
import re
import numpy as np
from google.colab import drive # Import library needed for Colab/Drive
import os

# ----------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# ----------------------------------------------------
# This will prompt you to authorize Colab to access your Google Drive.
print("Mounting Google Drive...")
drive.mount('/content/drive')

# Define the file paths for the new dataset
FILE_NAME = 'Ref002_Title_Split.csv'
OUTPUT_FILE_NAME = 'Titles_Split_Ref002.csv'

FILE_PATH = f'/content/drive/MyDrive/连环画/Ref002/{FILE_NAME}'
OUTPUT_PATH = f'/content/drive/MyDrive/连环画/Ref002/{OUTPUT_FILE_NAME}'

# ----------------------------------------------------
# 2. DEFINE PROCESSING FUNCTION
# ----------------------------------------------------

def process_titles_corrected(title):
    """
    Applies conditional splitting logic based on the FIRST separator (em dash '—' or left bracket '（').
    Returns (Title-Core, Title-Extra).
    """
    if pd.isna(title) or not isinstance(title, str):
        return title, None

    title = title.strip()

    # Find the index of the first em dash
    idx_emdash = title.find('—')
    # Find the index of the first left bracket
    idx_bracket = title.find('（')

    # Determine the actual split point (the first one found)
    split_index = -1
    separator = None

    # Prioritize the first occurrence of either '—' or ''
    if idx_emdash != -1 and (idx_bracket == -1 or idx_emdash < idx_bracket):
        split_index = idx_emdash
        separator = '—'
    elif idx_bracket != -1 and (idx_emdash == -1 or idx_bracket < idx_emdash):
        split_index = idx_bracket
        separator = '（'

    # If no separator found, return the full title as core
    if split_index == -1:
        return title, None

    # --- Apply Conditional Splitting Rules ---
    core = title[:split_index].strip()

    if separator == '—':
        # Rule: Exclude the em dash from Title-Extra
        extra = title[split_index + len('—'):].strip()
    elif separator == '（': # separator == '('
        # Rule: Include the left bracket in Title-Extra
        extra = title[split_index:].strip()

    # Ensure 'extra' is not empty after stripping
    if not extra:
        return title, None # No actual extra info, treat as unsplit

    return core, extra


def apply_processing_to_df(df, title_column='Title'):
    """
    Applies the corrected split function to the DataFrame and creates the check column.
    """
    # Apply the split function row-wise
    df[['Title-Core', 'Title-Extra']] = df[title_column].apply(
        lambda x: pd.Series(process_titles_corrected(x))
    )

    # Create the 'Check-Titles' column
    # TRUE if a split successfully created a Title-Extra value (i.e., Title-Extra is NOT null)
    df['Check-Titles'] = df['Title-Extra'].notna()

    return df

# ----------------------------------------------------
# 3. EXECUTION: Load, Process, and Save
# ----------------------------------------------------
try:
    # Load the data from Google Drive
    df = pd.read_csv(FILE_PATH)
    print(f"\nSuccessfully loaded {len(df)} rows from: {FILE_PATH}")

    # Apply the processing function
    df_processed = apply_processing_to_df(df, title_column='Title')

    # Save the processed data back to Google Drive
    df_processed.to_csv(OUTPUT_PATH, index=False)

    print(f"\nProcessing Complete. {df_processed['Check-Titles'].sum()} titles were split.")
    print(f"Results saved to: {OUTPUT_PATH}")

    # Verification (First 5 Rows of the new columns)
    print("\nVerification (First 5 Rows with new/updated columns):")
    print(df_processed[['Title', 'Title-Core', 'Title-Extra', 'Check-Titles']].head(15).to_markdown(index=False))

except FileNotFoundError:
    print(f"\nERROR: File not found at the specified path: {FILE_PATH}")
    print("Please check that the file is uploaded to your Google Drive and the FILE_PATH variable is correct.")
except Exception as e:
    print(f"\nAn error occurred during processing: {e}")

# ----------------------------------------------------
# 4. IDENTIFY POTENTIAL SPLIT CANDIDATES
# ----------------------------------------------------
print("\n" + "="*50)
print("CANDIDATE IDENTIFICATION (Unsplit Titles with Delimiters)")
print("="*50)

try:
    # Filter the DataFrame to include only rows that were NOT split
    unprocessed_df = df_processed[df_processed['Check-Titles'] == False]

    # Regex to find common alternative separators (colon, hyphen, pipe, square/curly brackets)
    # Note: This regex was originally for a different set of delimiters and is kept as-is
    # as the request focused only on the primary splitting logic.
    REGEX_CANDIDATE = r'[\:\-\|[\[\{]）-，；：'

    # Identify candidates in the unprocessed subset
    candidate_mask = unprocessed_df['Title'].str.contains(REGEX_CANDIDATE, na=False)
    candidate_titles = unprocessed_df.loc[candidate_mask, 'Title'].tolist()

    if candidate_titles:
        print(f"Found {len(candidate_titles)} titles that contain alternate delimiters:")
        # Print only the first 10 candidates for brevity
        for title in candidate_titles[:10]:
            print(f"- {title}")
        if len(candidate_titles) > 10:
             print(f"... and {len(candidate_titles) - 10} more titles.")
    else:
        print("No additional candidates found based on alternative delimiters \:\-\|[\[\{]）-，；：")

except Exception as e:
    print(f"Error during candidate identification: {e}")

<>:143: SyntaxWarning: invalid escape sequence '\:'
<>:143: SyntaxWarning: invalid escape sequence '\:'
/tmp/ipykernel_887/2168699025.py:143: SyntaxWarning: invalid escape sequence '\:'
  print("No additional candidates found based on alternative delimiters \:\-\|[\[\{]）-，；：")


Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Successfully loaded 2796 rows from: /content/drive/MyDrive/连环画/Ref002/Ref002_Title_Split.csv

Processing Complete. 255 titles were split.
Results saved to: /content/drive/MyDrive/连环画/Ref002/Titles_Split_Ref002.csv

Verification (First 5 Rows with new/updated columns):
| Title              | Title-Core         | Title-Extra   | Check-Titles   |
|:-------------------|:-------------------|:--------------|:---------------|
| 一支驳壳枪         | 一支驳壳枪         |               | False          |
| 一张奇怪的药方     | 一张奇怪的药方     |               | False          |
| 一块银元           | 一块银元           |               | False          |
| 一颗红心献人民     | 一颗红心献人民     |               | False          |
| 一块风化石         | 一块风化石         |               | False          |
| 一匹枣骝马         | 一匹枣骝马         |               | False          |
| 一把火             | 一把火   

In [ ]:
# Save the processed data back to Google Drive
# Ensure the output file name is defined. We can reuse OUTPUT_PATH from the first cell.
# OUTPUT_FILE_NAME and OUTPUT_PATH should be defined in the first cell or a preceding cell

try:
    df_processed.to_csv(OUTPUT_PATH, index=False)
    print(f"\nProcessed data saved successfully to: {OUTPUT_PATH}")
except NameError:
    print("Error: OUTPUT_PATH is not defined. Please ensure the cell defining file paths has been run.")
except Exception as e:
    print(f"\nAn error occurred while saving the file: {e}")


Processed data saved successfully to: /content/drive/MyDrive/连环画/Ref002/Titles_Split_Ref002.csv


In [ ]:
import pandas as pd
import re
import numpy as np
from google.colab import drive
import os

# Define the file paths
FILE_NAME = 'Titles_Split_Ref002.csv'
OUTPUT_FILE_NAME = 'Ref002_Candidates.csv'
# Assuming the file is in the root of your Google Drive:
FILE_PATH = f'/content/drive/MyDrive/连环画/Ref002/{FILE_NAME}'
OUTPUT_PATH = f'/content/drive/MyDrive/{OUTPUT_FILE_NAME}'

# ----------------------------------------------------
# 2. DEFINE CANDIDATE REGEX
# ----------------------------------------------------

# This regex looks for a comprehensive set of common Chinese and English punctuation marks.
# It specifically excludes the space (' ') and left parenthesis ('(') which were the original split criteria.
REGEX_PUNCTUATION = r'[\:\-\—\.\/，、。？！「」『』【】（）\[\]\{\}\|\/]'
# Breakdown: Colon, Hyphen, Em-dash, Dot, Slash, and various Chinese/Full-width punctuation marks.

# ----------------------------------------------------
# 3. EXECUTION: Load and Process
# ----------------------------------------------------

try:
    # Load the data from Google Drive
    # Ensure 'Title-Extra' is treated as a string to correctly handle empty values/NaNs
    df = pd.read_csv(FILE_PATH, dtype={'Title-Extra': str})
    print(f"\nSuccessfully loaded {len(df)} rows from: {FILE_PATH}")

    # Initialize the new column with None/NaN (Python's default empty state)
    df['Title-Punctuation'] = None

    # --- Step A: Identify Unsplit Titles ---
    # Rows where 'Title-Extra' is empty (NaN, None, or just whitespace).
    unsplit_mask = df['Title-Extra'].fillna('').str.strip() == ''

    # --- Step B: Identify Punctuation Candidates within Unsplit Titles ---
    # Apply the regex only to the 'Title' column of the unsplit subset.
    punctuation_mask = df.loc[unsplit_mask, 'Title'].str.contains(REGEX_PUNCTUATION, na=False)

    # --- Step C: Populate the new column ---
    # Get the indices of the rows that are both UNSPLIT AND contain PUNCTUATION
    candidate_indices = punctuation_mask[punctuation_mask].index

    # Populate 'Title-Punctuation' with the original 'Title' string for these candidates
    df.loc[candidate_indices, 'Title-Punctuation'] = df.loc[candidate_indices, 'Title']

    # --- Step D: Save the results ---
    df.to_csv(OUTPUT_PATH, index=False)

    total_candidates = df['Title-Punctuation'].count()

    print(f"\nProcessing Complete.")
    print(f"Total entries analyzed: {len(df)}")
    print(f"Total titles identified with punctuation that were previously unsplit: {total_candidates}")
    print(f"Full DataFrame saved to: {OUTPUT_PATH}")

    # Verification (All candidates)
    print("\nVerification (All Entries where 'Title-Punctuation' is not empty):")
    if total_candidates > 0:
        print(df[df['Title-Punctuation'].notna()][['Title', 'Title-Extra', 'Title-Punctuation']].to_markdown(index=False))
    else:
        print("No candidates found that meet the criteria.")

except FileNotFoundError:
    print(f"\nERROR: File not found at the specified path: {FILE_PATH}")
    print("Please check that the file is uploaded to your Google Drive and the FILE_PATH variable is correct.")
except Exception as e:
    print(f"\nAn error occurred during processing: {e}")


Successfully loaded 2796 rows from: /content/drive/MyDrive/连环画/Ref002/Titles_Split_Ref002.csv

Processing Complete.
Total entries analyzed: 2796
Total titles identified with punctuation that were previously unsplit: 7
Full DataFrame saved to: /content/drive/MyDrive/Ref002_Candidates.csv

Verification (All Entries where 'Title-Punctuation' is not empty):
| Title             |   Title-Extra | Title-Punctuation   |
|:------------------|--------------:|:--------------------|
| 谁是“螳螂”？      |           nan | 谁是“螳螂”？        |
| 战斗在九938.2高地 |           nan | 战斗在九938.2高地   |
| 自豪吧，母亲!     |           nan | 自豪吧，母亲!       |
| 爱情啊，你姓什么  |           nan | 爱情啊，你姓什么    |
| 我、你、他        |           nan | 我、你、他          |
| “橡树，十万火急”  |           nan | “橡树，十万火急”    |
| 啊，野麦岭        |           nan | 啊，野麦岭          |
